# Proyecto Final - Deep Learning (1INF52)
## Entregable 2: Fine-tuning de T5 para generación de resúmenes

**Integrantes:**

Aarón Ulises Santillán Huamán - 20200445

Joseph Alvaro Tintaya Marce - 20201155

Leoncio Percy Huaman Lopez - 20191833

Rodrigo Jose Valle Saavedra - 20210680

> Este cuaderno implementa el pipeline de fine-tuning del modelo T5 sobre el dataset CNN/DailyMail
usando la librería Hugging Face. Incluye carga de datos, preprocesamiento, entrenamiento,
evaluación con métricas ROUGE y guardado del modelo.

## 1. Importaciones y configuración inicial

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q transformers datasets sentencepiece evaluate rouge_score accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
from datasets import load_dataset
from transformers import (
    T5ForConditionalGeneration,
    T5TokenizerFast,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)
import evaluate

# Configuración de dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")

Dispositivo: cuda


In [ ]:
# Seed
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
if device == "cuda":
    torch.cuda.manual_seed(seed)

In [ ]:
import transformers
print("Transformers version:", transformers.__version__)

Transformers version: 4.57.1


## 2. Carga del dataset CNN/DailyMail

Usaremos la versión `3.0.0` del dataset `cnn_dailymail` disponible en Hugging Face Datasets.

In [ ]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

# Reducción de la data original
dataset["train"] = dataset["train"].select(range(20000))
dataset["validation"] = dataset["validation"].select(range(2000))
dataset["test"] = dataset["test"].select(range(2000))

print(dataset["train"][0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

## 3. Carga del tokenizer y del modelo T5

In [ ]:
model_name = 't5-small'

tokenizer = T5TokenizerFast.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.gradient_checkpointing_enable()
model.to(device)

model.train()


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

## 4. Preprocesamiento de datos
Convertimos cada ejemplo en pares `(input_ids, labels)` para el modelo T5.

- Entrada: prefijo `summarize: ` + artículo
- Salida: `highlights` (resumen humano)


In [ ]:
max_input_length =512
max_target_length = 128

def preprocess_function(batch):
    inputs = ["summarize: " + doc for doc in batch['article']]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding='max_length',
    )

    labels = tokenizer(
        text_target=batch["highlights"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length",
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


In [ ]:
small_train = dataset['train'].shuffle(seed=seed).select(range(10000))
small_val = dataset['validation'].shuffle(seed=seed).select(range(1000))

tokenized_train = small_train.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset['train'].column_names,
)
tokenized_val = small_val.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset['validation'].column_names,
)
tokenized_train, tokenized_val

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

(Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 10000
 }),
 Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 1000
 }))

## 5. Definición de la métrica ROUGE

In [ ]:
rouge = evaluate.load('rouge')

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]
    return preds, labels

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )
    # Dejar en porcentaje
    result = {k: round(v * 100, 2) for k, v in result.items()}
    return result


## 6. Configuración de entrenamiento (TrainingArguments + Trainer)

In [ ]:
from transformers import (
    T5ForConditionalGeneration,
    T5TokenizerFast,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)

output_dir = "./t5_cnn_summarization"

num_epochs = 3


data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adafactor",
    fp16=True,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=num_epochs,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=False,
    remove_unused_columns=False,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=None,
)


/tmp/ipython-input-531196102.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## 7. Entrenamiento del modelo

In [ ]:
train_result = trainer.train()
trainer.save_model(output_dir)
train_result

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: a20201155 (a20201155-pontificia-universidad-cat-lica-del-per-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
100,1.794100
200,1.112000
300,1.099800
400,1.133400
500,1.073200
600,1.060400
700,1.121500
800,1.078200
900,1.088000
1000,1.064700


TrainOutput(global_step=7500, training_loss=1.0268635681152343, metrics={'train_runtime': 3284.3466, 'train_samples_per_second': 9.134, 'train_steps_per_second': 2.284, 'total_flos': 4060254044160000.0, 'train_loss': 1.0268635681152343, 'epoch': 3.0})

## 8. Evaluación en el conjunto de validación

In [ ]:
from transformers import T5ForConditionalGeneration, T5TokenizerFast
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
import torch
from tqdm import tqdm
from evaluate import load

In [ ]:
# Cargar modelo y tokenizer

output_dir = "./t5_cnn_summarization"

tokenizer = T5TokenizerFast.from_pretrained(output_dir)
model = T5ForConditionalGeneration.from_pretrained(output_dir)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("Modelo cargado en:", device)

Modelo cargado en: cuda


In [ ]:
# Cargar dataset CNN/DailyMail
dataset = load_dataset("cnn_dailymail", "3.0.0")
val_dataset = dataset["validation"]

articles = ["summarize: " + a for a in val_dataset["article"]]
highlights = list(val_dataset["highlights"])

In [ ]:
# Tokenización del dataset

tokenized_val = tokenizer(
    articles,
    truncation=True,
    padding="max_length",
    max_length=512,
    return_tensors="pt"
)

labels_val = tokenizer(
    highlights,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

# Ajustar labels para ignorar padding
labels = labels_val["input_ids"]
labels[labels == tokenizer.pad_token_id] = -100

# Tensores
inputs = tokenized_val["input_ids"]
attn = tokenized_val["attention_mask"]

dataset_eval = TensorDataset(inputs, attn, labels)

In [ ]:
# DataLoader optimizado
loader = DataLoader(dataset_eval, batch_size=8, shuffle=False)

In [ ]:
# Generación

preds = []
refs = []

for batch in tqdm(loader):
    input_ids, attention_mask, label_ids = [t.to(device) for t in batch]

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=80,
            num_beams=4
        )

    preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    clean_labels = torch.where(label_ids == -100, torch.tensor(tokenizer.pad_token_id).to(device), label_ids)
    refs.extend(tokenizer.batch_decode(clean_labels, skip_special_tokens=True))

print("Predicciones generadas:", len(preds))

100%|██████████| 1671/1671 [36:37<00:00,  1.31s/it]

Predicciones generadas: 13368


In [ ]:
# Cálculo de ROUGE

rouge = load("rouge")
results = rouge.compute(predictions=preds, references=refs)

print("\n================== MÉTRICAS ROUGE ==================")
for k, v in results.items():
    print(f"{k}: {float(v):.4f}")
print("====================================================\n")


================== MÉTRICAS ROUGE ==================
rouge1: 0.3971
rouge2: 0.1841
rougeL: 0.2825
rougeLsum: 0.2824



## 9. Ejemplos de resúmenes generados

In [ ]:
def generate_summary(article, max_new_tokens=128):
    # Construimos el prompt para T5
    input_text = "summarize: " + article
    inputs = tokenizer(
        input_text,
        return_tensors='pt',
        truncation=True,
        max_length=max_input_length,
    ).to(device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        early_stopping=True,
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Tomamos un ejemplo del conjunto de validación original (no tokenizado)
example = dataset['validation'][0]
print('=== ARTÍCULO ORIGINAL ===')
print(example['article'][:1000], '...')

print('\n=== RESUMEN HUMANO (highlights) ===')
print(example['highlights'])

print('\n=== RESUMEN GENERADO POR EL MODELO ===')
generated = generate_summary(example['article'])
print(generated)


=== ARTÍCULO ORIGINAL ===
(CNN)Share, and your gift will be multiplied. That may sound like an esoteric adage, but when Zully Broussard selflessly decided to give one of her kidneys to a stranger, her generosity paired up with big data. It resulted in six patients receiving transplants. That surprised and wowed her. "I thought I was going to help this one person who I don't know, but the fact that so many people can have a life extension, that's pretty big," Broussard told CNN affiliate KGO. She may feel guided in her generosity by a higher power. "Thanks for all the support and prayers," a comment on a Facebook page in her name read. "I know this entire journey is much bigger than all of us. I also know I'm just the messenger." CNN cannot verify the authenticity of the page. But the power that multiplied Broussard's gift was data processing of genetic profiles from donor-recipient pairs. It works on a simple swapping principle but takes it to a much higher level, according to Californ